In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
import numpy as np
import matplotlib.pyplot as plt

import os
import hashlib
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 25          # maximum; EarlyStopping below may stop sooner
SEED = 42           

# seeds python, numpy and TensorFlow (weight init, augmentation, shuffling)
tf.keras.utils.set_random_seed(SEED)

print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/gdrive/', force_remount=True)
    os.chdir('/content/gdrive/MyDrive/Tomato-disease/training/')
except ImportError:
    print("Not running on Colab - using current directory:", os.getcwd())

In [ ]:
dataset = tf.keras.utils.image_dataset_from_directory(
    "plantvillage",
    shuffle=True,
    seed=SEED,
    image_size = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size = BATCH_SIZE
)

In [ ]:
class_names = dataset.class_names
class_names

In [ ]:
len(dataset)   # number of batches, not images

In [ ]:
plt.figure(figsize = (15,10))
for image_batch, label_batch in dataset.take(1):
    for i in range(12):
        ax = plt.subplot(3,4, i+1)
        plt.title(class_names[label_batch[i].numpy()])
        plt.imshow(image_batch[i].numpy().astype("uint8"))   # images are 8-bit
        plt.axis("off")

In [ ]:
# Cache images as uint8 instead of float32 (4x less RAM: ~3 GB instead of ~12 GB for ~16k images at 256x256).
# The pipeline casts back to float32 before the model, so the model input is numerically the same as in the original.
dataset = dataset.map(
    lambda x, y: (tf.cast(tf.round(x), tf.uint8), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
def get_dataset_partitions_tf(ds, train_split=0.8, val_split=0.1, test_split=0.1, shuffle=True, shuffle_size=10000, seed=SEED):
    assert abs((train_split + val_split + test_split) - 1.0) < 1e-6, "splits must sum to 1"   # guard bad ratios
    ds_size = len(ds)

    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=seed, reshuffle_each_iteration=False)

    
    ds = ds.cache()
    for _ in ds:
        pass

    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)

    train_ds = ds.take(train_size) # 80% of dataset
    
    val_ds = ds.skip(train_size).take(val_size) # 10% of remaining 20% of dataset
    test_ds = ds.skip(train_size).skip(val_size) # other remaining 10% of dataset
    
    return train_ds, val_ds, test_ds

In [ ]:
train_ds, val_ds, test_ds = get_dataset_partitions_tf(dataset)
print("batches  train/val/test:", len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# leakage check. Hashes every image and counts images shared between splits. Expect 0 for all three
# (a few are possible only if the dataset itself contains byte-identical duplicate files).
def image_hashes(ds):
    hashes = set()
    for images, _ in ds:
        for img in images.numpy():
            hashes.add(hashlib.md5(img.tobytes()).hexdigest())
    return hashes

h_train, h_val, h_test = image_hashes(train_ds), image_hashes(val_ds), image_hashes(test_ds)
print("unique images  train/val/test:", len(h_train), len(h_val), len(h_test))
print("val  & train:", len(h_val & h_train))
print("test & train:", len(h_test & h_train))
print("val  & test :", len(h_val & h_test))

In [ ]:
def to_float(x, y):
    return tf.cast(x, tf.float32), y      # back to float32 (0-255), exactly what the original model received

train_ds = train_ds.unbatch().shuffle(1000, seed=SEED).batch(BATCH_SIZE).map(to_float).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.map(to_float).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.map(to_float).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
resize_and_rescale = tf.keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1.0/255.0)
])

In [ ]:
# augmentation layers are only active during training (fit); they are automatically off in evaluate/predict.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2)
])

Model Architecture

In [ ]:
input_shape = (None, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)   
n_classes = len(class_names)                             

model = models.Sequential([
    resize_and_rescale,
    data_augmentation,
    
    layers.Conv2D(32, (3,3), activation='relu'),    
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Flatten(),
    
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes, activation='softmax')
    
])

model.build(input_shape=input_shape)

model.summary()

Compiling and Training Model

In [ ]:
model.compile(
    optimizer='adam',
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics = ['accuracy']
)

In [ ]:
# EarlyStopping with restore_best_weights. The weights from the best validation-loss epoch are kept.
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# model training
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    verbose=1,
    validation_data=val_ds,
    callbacks=[early_stopping]
)

In [ ]:
scores = model.evaluate(test_ds)
scores   # [test loss, test accuracy]

In [ ]:
history.history.keys()

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
epochs_ran = range(len(acc))     # FIX: was range(EPOCHS), which crashes as soon as training stops early

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_ran, acc, label='Training Accuracy')
plt.plot(epochs_ran, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# FIX: the original computed loss/val_loss and sized the figure for two panels but never drew the loss panel
plt.subplot(1, 2, 2)
plt.plot(epochs_ran, loss, label='Training Loss')
plt.plot(epochs_ran, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# IMPROVEMENT: per-class evaluation. Overall accuracy hides weak classes when class sizes are uneven
# (check the per-class image counts of your copy of the dataset). Look at recall per class and at what gets confused with what.
y_true, y_pred = [], []
for images_batch, labels_batch in test_ds:
    y_true.extend(labels_batch.numpy())
    y_pred.extend(np.argmax(model.predict(images_batch, verbose=0), axis=1))

all_labels = list(range(n_classes))
print(classification_report(y_true, y_pred, labels=all_labels, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
plt.figure(figsize=(9, 9))
plt.imshow(cm, cmap='Blues')
plt.xticks(all_labels, class_names, rotation=90)
plt.yticks(all_labels, class_names)
for i in all_labels:
    for j in all_labels:
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix (test set)')
plt.tight_layout()
plt.show()

In [ ]:
for images_batch, labels_batch in test_ds.take(1):
    first_image = images_batch[0].numpy().astype('uint8')
    first_label = labels_batch[0].numpy()

    print("first image to predict")
    plt.imshow(first_image)
    print("actual label", class_names[first_label])

    batch_prediction = model.predict(images_batch, verbose=0)
    print("predicted label", class_names[np.argmax(batch_prediction[0])])

In [ ]:
def predict(model, img):
    # FIX: the original ignored the `img` argument and read the global `images[i]` (worked only inside the loop below).
    img_array = tf.keras.utils.img_to_array(img)     # (tf.keras.preprocessing.image.img_to_array is removed in Keras 3)
    img_array = tf.expand_dims(img_array, 0) # create a batch

    predictions = model.predict(img_array, verbose=0)

    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * float(np.max(predictions[0])), 2)
    return predicted_class, confidence

In [ ]:
plt.figure(figsize=(15, 15))      # IMPROVEMENT: the original had no figure size, so the 3-line titles overlapped
for images, labels in test_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        predicted_class, confidence =  predict(model, images[i].numpy())
        actual_class = class_names[labels[i].numpy()]

        plt.title(f"Actual: {actual_class}, \n Predicted: {predicted_class}, Confidence: {confidence}")
    
        plt.axis("off")

In [ ]:
# FIX: the original hard-coded model_version = 1 (overwriting it on every run) and its commented-out auto-version line crashes on an
#      empty or non-numeric ../models folder. This implements the intended auto-increment safely.
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)
existing_versions = [int(d) for d in os.listdir(models_dir) if d.isdigit()]
model_version = max(existing_versions, default=0) + 1

# FIX: on Keras 3 (TF >= 2.16) model.save("<folder>") raises an error; model.export() writes the SavedModel
#      (saved_model.pb) into ../models/<version>, the folder layout TensorFlow Serving expects.
saved_model_dir = f"{models_dir}/{model_version}"
if hasattr(model, "export"):
    model.export(saved_model_dir)
else:                                  # TF < 2.13
    model.save(saved_model_dir)

# IMPROVEMENT: also keep a native .keras file (reloadable with tf.keras.models.load_model for further training / analysis).
model.save(f"{models_dir}/tomato_v{model_version}.keras")
print("saved model version", model_version)

## Good to know (deliberately NOT changed, to stay faithful to the original design)

- **Serving contract:** the exported model takes raw pixels in 0-255, shape `(N, 256, 256, 3)`, float32. `Rescaling` is inside the model, so the API must not divide by 255 or normalise again.
- **Architecture:** no BatchNorm / Dropout. If validation loss climbs while training loss keeps falling, those (and stronger augmentation such as `RandomZoom`, `RandomContrast`, `RandomBrightness`) are the first things to try.
- **Class imbalance:** if per-class recall in the report above is uneven, pass `class_weight=` to `model.fit`.
- **Dataset bias:** PlantVillage photos are lab shots on plain backgrounds, often several per leaf. Even after this fix, treat the test score as an upper bound and try phone photos from the field.
- **Batch-level split:** the split is by batch of 32, so the ratio is only approximately 80/10/10 and the final partial batch lands in a random split.
- **Memory:** the frozen dataset lives in RAM (uint8). If the runtime runs out of memory, lower it by using a smaller `IMAGE_SIZE` or a subset of classes.